# CMT Convexity Correction Validation — Pucci (2014)

**Reference:** M. Pucci, *Constant Maturity Treasury Convexity Correction*, IJTAF 17(8), 2014. SSRN 3387961.

Visual validation of the credit-aware CMT convexity correction with three payoff variants (A, B, C).

## Contents
1. CC vs volatility — all three variants
2. CC vs hazard rate — credit dimension
3. No-default limit — collapse to Pelsser/Hagan CMS formula
4. Vega sign verification

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import math
import numpy as np
import matplotlib.pyplot as plt
from pricebook.cmt import cmt_convexity_corrections, cmt_cc_no_default
from pricebook.credit_adjustment import risky_annuity, risky_swap_rate

# 5Y fixing, 10Y annual bond, flat rate 4%
TS = 5.0; N = 10; FLAT_RATE = 0.04
YFS = [1.0] * N
TIMES = [TS + (i+1) for i in range(N)]
RF_DFS = [math.exp(-FLAT_RATE * t) for t in TIMES]
RF_DF_TS = math.exp(-FLAT_RATE * TS)
RF_DF_TP = math.exp(-FLAT_RATE * (TS + 1))

plt.rcParams.update({"figure.figsize": (10, 6), "font.size": 12})
print("Setup complete.")

## 1. CC vs Volatility — Three Variants

$\text{CC}^{(A)} = \text{CC}^{(B)}$ (Eq 34), while $\text{CC}^{(C)}$ includes the fix-or-pre-default term (Eq 35).

At $\sigma = 0$, all three vanish. Positive vega for typical payment conventions.

In [ ]:
sigmas = np.linspace(0.01, 0.50, 50)
gamma = 0.01  # 100bp hazard

# Compute R_cmt for this gamma
cra_dfs = [d * math.exp(-gamma * t) for d, t in zip(RF_DFS, TIMES)]
cra_Ts = RF_DF_TS * math.exp(-gamma * TS)
cra_Tn = cra_dfs[-1]
R_CMT = risky_swap_rate(cra_Ts, cra_Tn, risky_annuity(YFS, cra_dfs))

cc_a, cc_c = [], []
for sig in sigmas:
    r = cmt_convexity_corrections(R_CMT, sig, gamma, TS, YFS, RF_DFS, RF_DF_TS, RF_DF_TP)
    cc_a.append(r.cc_A * 10000)
    cc_c.append(r.cc_C * 10000)

fig, ax = plt.subplots()
ax.plot(sigmas * 100, cc_a, "b-", lw=2, label="CC$^{(A)}$ = CC$^{(B)}$")
ax.plot(sigmas * 100, cc_c, "r--", lw=2, label="CC$^{(C)}$ (fix-or-pre-default)")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("CMT Vol $\\sigma$ (%)")
ax.set_ylabel("CC (bp)")
ax.set_title(f"CMT Convexity Correction vs Vol ($\\gamma$={gamma*10000:.0f}bp, $T_s$={TS:.0f}Y)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. CC vs Hazard Rate — Credit Dimension

Higher hazard $\gamma$ increases the gap between CC$^{(C)}$ and CC$^{(A/B)}$.

At $\gamma = 0$, all three collapse to the Pelsser/Hagan CMS formula (Eq 37).

In [ ]:
gammas = np.linspace(0.0, 0.05, 50)  # 0 to 500bp hazard
sigma = 0.20

cc_a_g, cc_c_g = [], []
for g in gammas:
    cra_dfs_g = [d * math.exp(-g * t) for d, t in zip(RF_DFS, TIMES)]
    cra_Ts_g = RF_DF_TS * math.exp(-g * TS)
    cra_Tn_g = cra_dfs_g[-1]
    R_g = risky_swap_rate(cra_Ts_g, cra_Tn_g, risky_annuity(YFS, cra_dfs_g))
    r = cmt_convexity_corrections(R_g, sigma, g, TS, YFS, RF_DFS, RF_DF_TS, RF_DF_TP)
    cc_a_g.append(r.cc_A * 10000)
    cc_c_g.append(r.cc_C * 10000)

# Pelsser limit (gamma=0)
rf_ann = sum(y * d for y, d in zip(YFS, RF_DFS))
pelsser = cmt_cc_no_default(sigma, TS, 1.0/sum(YFS), rf_ann, RF_DF_TP) * 10000

fig, ax = plt.subplots()
ax.plot(gammas * 10000, cc_a_g, "b-", lw=2, label="CC$^{(A)}$ = CC$^{(B)}$")
ax.plot(gammas * 10000, cc_c_g, "r--", lw=2, label="CC$^{(C)}$")
ax.axhline(pelsser, color="gray", ls=":", lw=1.5, label=f"Pelsser/Hagan ($\\gamma$=0) = {pelsser:.1f}bp")
ax.set_xlabel("Hazard Rate $\\gamma$ (bp)")
ax.set_ylabel("CC (bp)")
ax.set_title(f"CMT CC vs Hazard Rate ($\\sigma$={sigma*100:.0f}%, $T_s$={TS:.0f}Y)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. No-Default Limit — Pelsser/Hagan Regression

At $\gamma = 0$: $\hat{D} = D$, $\hat{A} = A$, $\chi_0 = 1$, and all three CCs collapse to Eq (37):

$\text{CC} = (e^{\sigma^2 T_s} - 1)\left(1 - \alpha \frac{A_0}{D_{0,T_p}}\right)$

This is the cleanest regression test against any classical CMS pricer.

In [ ]:
# No-default: gamma=0, verify all 3 CCs = Pelsser
sigmas_nd = np.linspace(0.05, 0.50, 30)
cc_a_nd, cc_c_nd, pelsser_nd = [], [], []

rf_ann = sum(y * d for y, d in zip(YFS, RF_DFS))
alpha = 1.0 / sum(YFS)

R_rf = risky_swap_rate(RF_DF_TS, RF_DFS[-1], rf_ann)

for sig in sigmas_nd:
    r = cmt_convexity_corrections(R_rf, sig, 0.0, TS, YFS, RF_DFS, RF_DF_TS, RF_DF_TP)
    cc_a_nd.append(r.cc_A * 10000)
    cc_c_nd.append(r.cc_C * 10000)
    pelsser_nd.append(cmt_cc_no_default(sig, TS, alpha, rf_ann, RF_DF_TP) * 10000)

fig, ax = plt.subplots()
ax.plot(sigmas_nd * 100, cc_a_nd, "b-", lw=2, label="CC$^{(A)}$ ($\\gamma$=0)")
ax.plot(sigmas_nd * 100, cc_c_nd, "r--", lw=2, label="CC$^{(C)}$ ($\\gamma$=0)")
ax.plot(sigmas_nd * 100, pelsser_nd, "k:", lw=3, label="Pelsser/Hagan (Eq 37)")
ax.set_xlabel("CMT Vol $\\sigma$ (%)")
ax.set_ylabel("CC (bp)")
ax.set_title("No-Default Limit: All Three CCs Collapse to Pelsser/Hagan")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Print max deviation
max_dev_a = max(abs(a - p) for a, p in zip(cc_a_nd, pelsser_nd))
max_dev_c = max(abs(c - p) for c, p in zip(cc_c_nd, pelsser_nd))
print(f"Max deviation: CC(A) = {max_dev_a:.6f}bp, CC(C) = {max_dev_c:.6f}bp")